# Pipeline de Inteligência Artificial: Ebola L-Polymerase 🧬💻

**Módulo Sênior: Deep Learning com Quantificação de Incerteza (MC Dropout)**

Este notebook foi otimizado para ser *Self-Contained* (tudo em um só arquivo) para rodar perfeitamente no Google Colab (utilize Runtime GPU), sem depender de montagem do Google Drive.

In [ ]:
!pip install rdkit tensorflow pandas numpy matplotlib

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem

print(f"TensorFlow Version: {tf.__version__}")

### Infraestrutura Matemática (Injeta Localmente)

In [ ]:
# 1. Confidence-Weighted MSE (Função de Perda)
@tf.keras.utils.register_keras_serializable(name="ConfidenceWeightedMSE")
def confidence_weighted_mse(y_true, y_pred, confidence_weights=None):
    error_sq = tf.square(y_true - y_pred)
    if confidence_weights is not None:
        weights = tf.cast(confidence_weights, dtype=tf.float32)
        weighted_error = error_sq * weights
        return tf.reduce_mean(weighted_error)
    return tf.reduce_mean(error_sq)

class CW_MSE_Layer(tf.keras.losses.Loss):
    def __init__(self, name="confidence_weighted_mse", **kwargs):
        super().__init__(name=name, **kwargs)
    def call(self, y_true, y_pred):
        true_values = tf.expand_dims(y_true[:, 0], axis=-1)
        confidence = tf.expand_dims(y_true[:, 1], axis=-1)
        return confidence_weighted_mse(true_values, y_pred, confidence)

# 2. Monte Carlo Dropout (Incerteza)
class MCDropout(layers.Dropout):
    def call(self, inputs, training=None):
        # Força o dropout a estar sempre ativo, inclusive na predição
        return super().call(inputs, training=True)

def build_mc_dropout_model(input_dim):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = layers.Dense(512, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = MCDropout(0.2)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = MCDropout(0.2)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = MCDropout(0.1)(x)
    outputs = layers.Dense(1, activation='linear')(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="Ebola_Polymerase_MCDropout")

print("Módulos matemáticos carregados com sucesso diretamente no ambiente!")

### Ingestão de Dados e Transformação Molecular (RDKit)

In [ ]:
def smiles_to_morgan(smiles, radius=2, nBits=1024):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
            return np.array(fp)
    except:
        pass
    return np.zeros((nBits,))

print("Extraindo Fingerprints moleculares...")
dummy_smiles = ["CC1=C(C=C(C=C1)O)C=O", "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"]
X_train = np.array([smiles_to_morgan(s) for s in dummy_smiles])

# Construindo y_train como [Valor_Real_EC50, Peso_Confianca]
y_train = np.array([[1.2, 1.0], [4.5, 0.5]])

print(f"Shape de X_train (Fingerprints): {X_train.shape}")
print(f"Shape de y_train (Valor, Peso): {y_train.shape}")

### Compilação e Treinamento da Rede Neural (CW-MSE)

In [ ]:
model = build_mc_dropout_model(input_dim=1024)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=CW_MSE_Layer())

history = model.fit(X_train, y_train, epochs=5, batch_size=2, verbose=1)
print("Treinamento inicializado perfeitamente!")

### Inferência e Quantificação de Incerteza (MC Dropout Sampler)

In [ ]:
def predict_with_uncertainty(model, x_input, num_samples=100):
    predictions = []
    for _ in range(num_samples):
        pred = model(x_input).numpy()
        predictions.append(pred)
    
    predictions = np.array(predictions)
    mean_pred = np.mean(predictions, axis=0)
    std_dev = np.std(predictions, axis=0)
    return mean_pred, std_dev

print("Realizando amostragem Monte Carlo para predição de incerteza...")
mean, std = predict_with_uncertainty(model, X_train, num_samples=50)
print(f"Predição Média M1: {mean[0][0]:.4f} +/- {std[0][0]:.4f}")
print(f"Predição Média M2: {mean[1][0]:.4f} +/- {std[1][0]:.4f}")